# Retrieval evaluation: 100 Thai health questions

Measures routing accuracy, retrieval latency, source leakage, and exports retrieved citations/context for clinical review. This notebook does not evaluate medical-answer correctness.

In [2]:
from pathlib import Path
import json, sys, time
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'agent').exists():
    ROOT = ROOT.parent
assert (ROOT / 'agent').exists(), 'Run from project root or notebooks/'
sys.path.insert(0, str(ROOT))

from agent.analyst import _source_ids_for_retrieval
from agent.rag_utils import retrieve_context

DATASET = ROOT / 'data/evals/retrieval_questions_100.json'
questions = json.loads(DATASET.read_text(encoding='utf-8'))
assert len(questions) == 100
pd.DataFrame(questions).groupby('expected_source_id').size()

⚠️ Local Mode: ใช้ Credentials จากไฟล์ .env


expected_source_id
diabetes        25
dyslipidemia    25
hypertension    25
kidney          25
dtype: int64

In [3]:
# First retrieval may be cold because BGE-M3 is loaded on demand.
# Set WARMUP=False to record the cold-start case; otherwise benchmarks warm retrieval.
WARMUP = True
if WARMUP:
    retrieve_context('warm up', k=1)

rows = []
for item in questions:
    routed_sources = _source_ids_for_retrieval({}, item['question'])
    started = time.perf_counter()
    result = retrieve_context(item['question'], k=3, source_ids=routed_sources)
    latency_ms = round((time.perf_counter() - started) * 1000, 1)
    citation_sources = [c.get('source_id') for c in result.citations]
    rows.append({
        **item,
        'routed_sources': routed_sources,
        'routing_pass': item['expected_source_id'] in routed_sources,
        'citation_sources': citation_sources,
        'source_leakage': any(s != item['expected_source_id'] for s in citation_sources),
        'citation_count': len(result.citations),
        'latency_ms': latency_ms,
        'context_preview': result.context[:500].replace('\n', ' '),
    })

results = pd.DataFrame(rows)
results.to_csv(ROOT / 'data/evals/retrieval_results.csv', index=False, encoding='utf-8-sig')
results.head()

,id,question,expected_source_id,category,routed_sources,routing_pass,citation_sources,source_leakage,citation_count,latency_ms,context_preview
0,lipid-001,LDL 178 สูงไหม ต้องทำอย่างไร,dyslipidemia,ldl_interpretation,[dyslipidemia],True,"[dyslipidemia, dyslipidemia]",False,2,86.4,[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
1,lipid-002,LDL-C 195 ควรพบแพทย์เมื่อไร,dyslipidemia,ldl_interpretation,[dyslipidemia],True,"[dyslipidemia, dyslipidemia]",False,2,70.6,[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
2,lipid-003,LDL 160 ถือว่าสูงหรือไม่,dyslipidemia,ldl_interpretation,[dyslipidemia],True,"[dyslipidemia, dyslipidemia]",False,2,70.8,[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
3,lipid-004,LDL 130 ต้องกินยาหรือเปล่า,dyslipidemia,ldl_interpretation,[dyslipidemia],True,"[dyslipidemia, dyslipidemia]",False,2,71.5,[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
4,lipid-005,ค่า LDL ควรเป็นเท่าไร,dyslipidemia,ldl_target,[dyslipidemia],True,[dyslipidemia],False,1,68.3,[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...


In [4]:
summary = pd.Series({
    'questions': len(results),
    'routing_accuracy_pct': round(100 * results.routing_pass.mean(), 1),
    'source_leakage_pct': round(100 * results.source_leakage.mean(), 1),
    'empty_citation_pct': round(100 * (results.citation_count == 0).mean(), 1),
    'latency_p50_ms': round(results.latency_ms.quantile(.50), 1),
    'latency_p95_ms': round(results.latency_ms.quantile(.95), 1),
    'latency_max_ms': round(results.latency_ms.max(), 1),
})
display(summary.to_frame('value'))
display(results.groupby(['expected_source_id', 'category']).agg(
    questions=('id', 'count'), routing_accuracy=('routing_pass', 'mean'),
    p50_ms=('latency_ms', 'median'), p95_ms=('latency_ms', lambda x: x.quantile(.95))
).round(3))

# Manually review failures and context relevance in retrieval_results.csv.
results.loc[(~results.routing_pass) | results.source_leakage | (results.citation_count == 0),
            ['id', 'question', 'expected_source_id', 'routed_sources', 'citation_sources', 'context_preview']]

,value
questions,100.0
routing_accuracy_pct,91.0
source_leakage_pct,14.0
empty_citation_pct,0.0
latency_p50_ms,70.8
latency_p95_ms,121.8
latency_max_ms,239.8


questions  routing_accuracy  \
expected_source_id category                                              
diabetes           cardiovascular_risk             1               1.0   
                   complication_screening          2               1.0   
                   definition                      1               1.0   
                   diagnosis                       2               1.0   
                   fasting                         1               1.0   
...                                              ...               ...   
kidney             monitoring                      1               1.0   
                   prevention                      1               1.0   
                   screening                       1               1.0   
                   staging                         1               1.0   
                   symptoms                        1               1.0   

                                           p50_ms   p95_ms  
expected_source_id category                                 
diabetes           cardiovascular_risk      73.00   73.000  
                   complication_screening   98.35  119.455  
                   definition              122.90  122.900  
                   diagnosis               100.50  125.250  
                   fasting                  85.90   85.900  
...                                           ...      ...  
kidney             monitoring               74.10   74.100  
                   prevention               66.30   66.300  
                   screening                70.60   70.600  
                   staging                  69.20   69.200  
                   symptoms                 68.40   68.400  

[61 rows x 4 columns]

,id,question,expected_source_id,routed_sources,citation_sources,context_preview
10,lipid-011,TG 250 สูงไหม,dyslipidemia,[],"[kidney, kidney, kidney]",[ข้อมูลที่ 1 จาก: แนวทางการดูแลผู้ป่วยโรคไตเรื...
11,lipid-012,ไตรกลีเซอไรด์สูงควรงดอาหารกี่ชั่วโมงก่อนตรวจซ้ำ,dyslipidemia,"[dyslipidemia, kidney]","[dyslipidemia, kidney, kidney]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
17,lipid-018,statin คือยาอะไร,dyslipidemia,[],"[dyslipidemia, dyslipidemia]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
18,lipid-019,กิน statin แล้วปวดกล้ามเนื้อควรทำอย่างไร,dyslipidemia,[],"[dyslipidemia, dyslipidemia]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติการบำบัดภาวะ...
21,lipid-022,ควรตรวจ lipid profile บ่อยแค่ไหน,dyslipidemia,[],"[kidney, kidney, kidney]",[ข้อมูลที่ 1 จาก: แนวทางการดูแลผู้ป่วยโรคไตเรื...
37,diabetes-013,เบาหวานควรตรวจไตอย่างไร,diabetes,"[diabetes, kidney]","[diabetes, kidney]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติสำหรับโรคเบา...
40,diabetes-016,metformin เป็นยาชนิดใด,diabetes,[],"[diabetes, kidney, diabetes]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติสำหรับโรคเบา...
54,kidney-005,ACR ในปัสสาวะคืออะไร,kidney,[],"[diabetes, kidney, kidney]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติสำหรับโรคเบา...
55,kidney-006,โปรตีนรั่วในปัสสาวะต้องตรวจอะไร,kidney,[],"[kidney, kidney]",[ข้อมูลที่ 1 จาก: แนวทางการดูแลผู้ป่วยโรคไตเรื...
57,kidney-008,ผู้ป่วยเบาหวานควรตรวจไตบ่อยแค่ไหน,kidney,"[diabetes, kidney]","[diabetes, kidney, diabetes]",[ข้อมูลที่ 1 จาก: แนวทางเวชปฏิบัติสำหรับโรคเบา...
